In [0]:
-- Owner: Maeve
-- Name: 25 - Analytics KPIs and Validation
-- Purpose: Build dashboard KPIs and perform sanity checks across all seven analytics tables.
-- Grain: One KPI summary row and one validation summary row per analytics table.

USE CATALOG workspace;
USE SCHEMA instacart_analytics;

-- Build the dashboard KPI summary
CREATE OR REPLACE TABLE analytics_kpis AS
SELECT
  COUNT(DISTINCT order_id) AS total_orders,
  COUNT(*) AS total_order_lines,
  COUNT(DISTINCT product_id) AS total_products,
  ROUND(
    AVG(CASE WHEN reordered = TRUE THEN 1.0 ELSE 0.0 END),
    4
  ) AS overall_reorder_rate
FROM workspace.instacart_gold.gold_fact_order_product;

-- Preview the KPI summary
SELECT *
FROM analytics_kpis;

-- Validate all seven analytics tables
WITH validation AS (
  SELECT
    'analytics_top_departments' AS table_name,
    COUNT(*) AS row_count,
    0 AS invalid_values
  FROM analytics_top_departments

  UNION ALL

  SELECT
    'analytics_top_products',
    COUNT(*),
    0
  FROM analytics_top_products

  UNION ALL

  SELECT
    'analytics_day_hour_patterns',
    COUNT(*),
    0
  FROM analytics_day_hour_patterns

  UNION ALL

  SELECT
    'analytics_basket_size_by_day',
    COUNT(*),
    0
  FROM analytics_basket_size_by_day

  UNION ALL

  SELECT
    'analytics_reorder_rates',
    COUNT(*),
    COUNT_IF(
      reorder_rate IS NULL
      OR reorder_rate < 0
      OR reorder_rate > 1
    )
  FROM analytics_reorder_rates

  UNION ALL

  SELECT
    'analytics_product_pairs',
    COUNT(*),
    0
  FROM analytics_product_pairs

  UNION ALL

  SELECT
    'analytics_kpis',
    COUNT(*),
    COUNT_IF(
      overall_reorder_rate IS NULL
      OR overall_reorder_rate < 0
      OR overall_reorder_rate > 1
      OR total_orders <= 0
      OR total_order_lines <= 0
      OR total_products <= 0
      OR total_orders > total_order_lines
      OR total_products > total_order_lines
    )
  FROM analytics_kpis
)
SELECT
  table_name,
  row_count,
  invalid_values,
  CASE
    WHEN row_count > 0
      AND invalid_values = 0
      AND (
        table_name <> 'analytics_kpis'
        OR row_count = 1
      )
    THEN 'PASS'
    ELSE 'REVIEW'
  END AS status
FROM validation
ORDER BY table_name;